# Asset Embeddings: Return Comovement and Managed-Portfolio Benchmarks

This notebook completes the two remaining empirical benchmarks in Gabaix, Koijen, Richmond, and Yogo, *Asset Embeddings*. It treats the completed 80-quarter replication as immutable. All new downloads and outputs are checkpointed separately.

The return-comovement benchmark uses quarter `t-1` embeddings to explain monthly returns in quarter `t`. The managed-portfolio benchmark uses a 90%/10% investor split and masks the second-largest position of each test investor. An exact managed-portfolio result requires models trained only on the training investors; the full-sample embeddings are not used for that final score.

## 1. Environment, paths, and reproducible settings

In [ ]:
from pathlib import Path
import gc, os, time, warnings
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.special import logsumexp
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name != 'Asset_Embeddings':
    candidate = PROJECT_ROOT / 'Projects' / 'Asset_Embeddings'
    if candidate.exists(): PROJECT_ROOT = candidate.resolve()
FULL_ROOT = PROJECT_ROOT / 'full_replication_80q'
BENCH_ROOT = PROJECT_ROOT / 'remaining_benchmarks'
RAW_RETURN_ROOT = BENCH_ROOT / 'data' / 'crsp_monthly_returns'
RC_ROOT = BENCH_ROOT / 'return_comovement'
MP_ROOT = BENCH_ROOT / 'managed_portfolio'
for path in [RAW_RETURN_ROOT, RC_ROOT, MP_ROOT]: path.mkdir(parents=True, exist_ok=True)

QUARTERS = [(y, q) for y in range(2005, 2025) for q in range(1, 5)]
MODELS = ['rs_binary', 'rs_ranks', 'rs_l_0', 'rs_l_min', 'rs_l', 'word2vec']
DIMENSIONS = [4, 10]
ALPHAS = np.logspace(-6, 6, 49)
FIRM_SPLIT_SEED = 202451
INVESTOR_SPLIT_SEED = 202451
FORCE_DOWNLOAD = False
FORCE_REBUILD = False

def qlabel(year, quarter): return f'{year}q{quarter}'
def next_quarter(year, quarter): return (year + (quarter == 4), 1 if quarter == 4 else quarter + 1)
def save_parquet(frame, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_parquet(path, index=False)

print('Project root:', PROJECT_ROOT)
print('Benchmark output root:', BENCH_ROOT)

## 2. Immutable-input audit

This cell confirms that all matrices and embeddings required by the remaining benchmarks exist. It does not modify the full-replication directory.

In [ ]:
audit_rows = []
for year, quarter in QUARTERS:
    label = qlabel(year, quarter)
    matrix_root = FULL_ROOT / 'matrices' / label
    row = {
        'year': year, 'quarter': quarter,
        'binary_matrix': (matrix_root / 'binary.npz').exists(),
        'log_matrix': (matrix_root / 'log_dollar_holdings_positive.npz').exists(),
        'firm_mapping': (matrix_root / 'firm_mapping.parquet').exists(),
        'investor_mapping': (matrix_root / 'investor_mapping.parquet').exists(),
        'characteristics': (FULL_ROOT / 'processed' / label / 'jkp_firm_panel.parquet').exists(),
    }
    embedding_paths = {
        'rs_binary': [FULL_ROOT / 'embeddings' / label / 'rs_binary' / 'asset_embeddings_normalized_labeled.parquet'],
        'rs_l_min': [FULL_ROOT / 'embeddings' / label / 'rs_l_min' / 'asset_embeddings_normalized_labeled.parquet'],
        'rs_ranks': [FULL_ROOT / 'additional_recommender_systems' / 'embeddings' / label / 'rs_ranks' / 'asset_embeddings_normalized_labeled.parquet'],
        'rs_l_0': [FULL_ROOT / 'additional_recommender_systems' / 'embeddings' / label / 'rs_l_0' / 'asset_embeddings_normalized_labeled.parquet'],
        'rs_l': [FULL_ROOT / 'additional_recommender_systems' / 'embeddings' / label / 'rs_l' / f'dimension={d}' / 'asset_embeddings_labeled.parquet' for d in DIMENSIONS],
        'word2vec': [FULL_ROOT / 'word2vec' / 'embeddings' / label / f'dimension={d}' / 'asset_embeddings_labeled.parquet' for d in DIMENSIONS],
    }
    for model, paths in embedding_paths.items():
        row[f'{model}_embedding'] = all(path.exists() for path in paths)
    audit_rows.append(row)
input_audit = pd.DataFrame(audit_rows)
bool_cols = input_audit.select_dtypes(include='bool').columns
input_audit['complete'] = input_audit[bool_cols].all(axis=1)
save_parquet(input_audit, BENCH_ROOT / 'input_audit.parquet')
display(input_audit.groupby('complete').size().rename('quarters').reset_index())
assert input_audit.complete.all(), 'Some immutable replication inputs are missing; inspect input_audit.'

## 3. Connect to WRDS

This connection is used only to obtain CRSP monthly returns. Existing local return files are reused automatically. Enter your WRDS credentials when prompted.

In [ ]:
import sys
local_packages = PROJECT_ROOT / '.packages'
if local_packages.exists() and str(local_packages) not in sys.path: sys.path.insert(0, str(local_packages))
import wrds
try:
    db.close()
except Exception:
    pass
db = wrds.Connection()
print('WRDS connection is active.')

## 4. Download and construct monthly firm returns

CRSP monthly security returns are joined to the date-valid name history to obtain `permco`. Multiple securities belonging to one firm are aggregated using lagged market capitalization. The local file covers 2005Q2 through 2025Q1 so that 2005Q1–2024Q4 embeddings can be paired with the following quarter where available.

In [ ]:
returns_path = RAW_RETURN_ROOT / 'crsp_firm_monthly_2004q1_2025q1.parquet'
if returns_path.exists() and not FORCE_DOWNLOAD:
    firm_returns = pd.read_parquet(returns_path)
    download_status = 'reused local file'
else:
    query = """
        select m.permno, n.permco, m.mthcaldt as date,
               m.mthret as ret, m.mthprevcap as lag_market_cap,
               n.shrcd, n.exchcd
        from crsp_a_stock.stkmthsecuritydata as m
        join crsp_a_stock.stocknames as n
          on m.permno = n.permno
         and m.mthcaldt between n.namedt and coalesce(n.nameenddt, '9999-12-31')
        where m.mthcaldt >= '2004-01-01'
          and m.mthcaldt < '2025-04-01'
          and n.shrcd in (10, 11)
    """
    raw = db.raw_sql(query, date_cols=['date'])
    for col in ['permno', 'permco', 'ret', 'lag_market_cap']:
        raw[col] = pd.to_numeric(raw[col], errors='coerce')
    raw = raw.dropna(subset=['permco', 'date', 'ret']).copy()
    raw['lag_market_cap'] = raw.lag_market_cap.where(raw.lag_market_cap > 0)
    raw['weight'] = raw.groupby(['permco', 'date']).lag_market_cap.transform(lambda x: x / x.sum())
    raw['weighted_return'] = raw.ret * raw.weight
    value_weighted = raw.dropna(subset=['weight']).groupby(['permco', 'date'], as_index=False).agg(ret=('weighted_return', 'sum'), securities=('permno', 'nunique'))
    equal_weighted = raw.groupby(['permco', 'date'], as_index=False).agg(equal_ret=('ret', 'mean'))
    firm_returns = equal_weighted.merge(value_weighted, on=['permco', 'date'], how='left')
    firm_returns['ret'] = firm_returns.ret.fillna(firm_returns.equal_ret)
    firm_returns['permco'] = firm_returns.permco.astype('int64')
    firm_returns['return_year'] = firm_returns.date.dt.year
    firm_returns['return_quarter'] = firm_returns.date.dt.quarter
    firm_returns = firm_returns[['permco', 'date', 'return_year', 'return_quarter', 'ret', 'securities']]
    save_parquet(firm_returns, returns_path)
    download_status = 'downloaded'
print(download_status, len(firm_returns), 'firm-month observations')
display(firm_returns.groupby(['return_year','return_quarter']).agg(months=('date','nunique'), firms=('permco','nunique'), observations=('ret','size')).reset_index().tail())

## 5. Construct the six return-benchmark characteristics

The paper compares embeddings with beta, log market equity, log book-to-market equity, asset growth, profitability, and momentum. Momentum is constructed strictly from returns preceding the embedding quarter-end; no outcome-quarter return enters a predictor.

In [ ]:
def quarter_end(year, quarter):
    return pd.Timestamp(year=year, month=quarter * 3, day=1) + pd.offsets.MonthEnd(0)

def trailing_momentum(return_frame, year, quarter):
    end = quarter_end(year, quarter) - pd.DateOffset(months=1)
    start = quarter_end(year, quarter) - pd.DateOffset(months=11)
    history = return_frame[return_frame.date.between(start, end)].copy()
    history = history[history.ret.gt(-1)].copy()
    momentum = history.groupby('permco').agg(
        momentum_months=('date', 'nunique'),
        momentum_12_1=('ret', lambda x: np.prod(1.0 + x) - 1.0),
    ).reset_index()
    return momentum[momentum.momentum_months >= 9]

characteristic_inventory = []
for year, quarter in QUARTERS:
    label = qlabel(year, quarter)
    output = RC_ROOT / 'characteristics' / f'{label}.parquet'
    if output.exists() and not FORCE_REBUILD:
        panel = pd.read_parquet(output)
    else:
        panel = pd.read_parquet(FULL_ROOT / 'processed' / label / 'jkp_firm_panel.parquet')
        panel = panel.merge(trailing_momentum(firm_returns, year, quarter), on='permco', how='left')
        panel['log_market_equity'] = np.log(panel.jkp_firm_market_equity.where(panel.jkp_firm_market_equity > 0))
        panel['log_book_to_market'] = np.log((panel.jkp_book_equity / panel.jkp_firm_market_equity).where((panel.jkp_book_equity > 0) & (panel.jkp_firm_market_equity > 0)))
        keep = ['permco','beta_60m_paper','log_market_equity','log_book_to_market','at_gr1_paper','gp_at_paper','momentum_12_1']
        panel = panel[keep].replace([np.inf, -np.inf], np.nan).dropna().copy()
        save_parquet(panel, output)
    characteristic_inventory.append({'year':year,'quarter':quarter,'complete_case_firms':len(panel)})
characteristic_inventory = pd.DataFrame(characteristic_inventory)
save_parquet(characteristic_inventory, RC_ROOT / 'characteristic_inventory.parquet')
display(characteristic_inventory.describe())

## 6. Return-comovement estimation helpers

For each transition, the same firm universe and 80/20 split are used for all models. Predictors are standardized using training firms only. Ridge penalties are chosen by ten-fold CV within each monthly training cross-section.

In [ ]:
def embedding_path(model, label):
    if model in ['rs_binary','rs_l_min']:
        return FULL_ROOT / 'embeddings' / label / model / 'asset_embeddings_normalized_labeled.parquet'
    if model in ['rs_ranks','rs_l_0']:
        return FULL_ROOT / 'additional_recommender_systems' / 'embeddings' / label / model / 'asset_embeddings_normalized_labeled.parquet'
    raise ValueError('Dimension-specific model path required')

def load_embedding(model, label, dimension):
    if model == 'rs_l':
        path = FULL_ROOT / 'additional_recommender_systems' / 'embeddings' / label / model / f'dimension={dimension}' / 'asset_embeddings_labeled.parquet'
    elif model == 'word2vec':
        path = FULL_ROOT / 'word2vec' / 'embeddings' / label / f'dimension={dimension}' / 'asset_embeddings_labeled.parquet'
    else:
        path = embedding_path(model, label)
    frame = pd.read_parquet(path)
    columns = sorted(c for c in frame if c.startswith('embedding_'))[:dimension]
    return frame[['permco'] + columns].rename(columns={c:f'{model}_{dimension}_{c}' for c in columns})

def ridge_fit_cv(X, y, seed):
    folds = KFold(n_splits=10, shuffle=True, random_state=seed)
    losses = np.zeros(len(ALPHAS), dtype='float64')
    for train, valid in folds.split(X):
        Xa = np.column_stack([np.ones(len(train)), X[train]])
        penalty = np.eye(Xa.shape[1]); penalty[0,0] = 0.0
        gram = Xa.T @ Xa; rhs = Xa.T @ y[train]
        Xv = np.column_stack([np.ones(len(valid)), X[valid]])
        for j, alpha in enumerate(ALPHAS):
            coefficient = np.linalg.solve(gram + alpha * penalty, rhs)
            losses[j] += np.mean((y[valid] - Xv @ coefficient) ** 2)
    best = int(np.argmin(losses))
    Xa = np.column_stack([np.ones(len(y)), X])
    penalty = np.eye(Xa.shape[1]); penalty[0,0] = 0.0
    coefficient = np.linalg.solve(Xa.T @ Xa + ALPHAS[best] * penalty, Xa.T @ y)
    return coefficient, float(ALPHAS[best])

CHARACTERISTICS = ['beta_60m_paper','log_market_equity','log_book_to_market','at_gr1_paper','gp_at_paper','momentum_12_1']

## 7. Run the 79-transition return-comovement benchmark

Each completed embedding-quarter/return-quarter transition is checkpointed. Restarting this cell reuses completed transitions.

In [ ]:
checkpoint_root = RC_ROOT / 'quarter_results'
checkpoint_root.mkdir(parents=True, exist_ok=True)
quarter_results = []
for year, quarter in QUARTERS:
    return_year, return_quarter = next_quarter(year, quarter)
    if return_year > 2024: continue
    label = qlabel(year, quarter); target_label = qlabel(return_year, return_quarter)
    checkpoint = checkpoint_root / f'{label}_to_{target_label}.parquet'
    if checkpoint.exists() and not FORCE_REBUILD:
        quarter_results.append(pd.read_parquet(checkpoint)); print(label, 'reused checkpoint'); continue
    started = time.perf_counter()
    panel = pd.read_parquet(RC_ROOT / 'characteristics' / f'{label}.parquet')
    predictor_sets = {'beta': ['beta_60m_paper'], 'characteristics_6': CHARACTERISTICS}
    for dimension in DIMENSIONS:
        placebo_columns = []
        rng = np.random.default_rng(FIRM_SPLIT_SEED + year * 10 + quarter + dimension)
        for j in range(dimension):
            column = f'placebo_{dimension}_{j+1:03d}'; panel[column] = rng.standard_normal(len(panel)); placebo_columns.append(column)
        predictor_sets[f'placebo_{dimension}'] = placebo_columns
        for model in MODELS:
            embedding = load_embedding(model, label, dimension)
            columns = [c for c in embedding if c != 'permco']
            panel = panel.merge(embedding, on='permco', how='inner', validate='one_to_one')
            predictor_sets[f'{model}_{dimension}'] = columns
            if dimension == 10: predictor_sets[f'{model}_{dimension}_plus_characteristics'] = columns + CHARACTERISTICS
    panel = panel.sort_values('permco').reset_index(drop=True)
    rng = np.random.default_rng(FIRM_SPLIT_SEED + year * 10 + quarter)
    test_count = int(np.ceil(0.20 * len(panel))); order = rng.permutation(len(panel))
    panel['is_test'] = False; panel.loc[order[:test_count], 'is_test'] = True
    outcome = firm_returns[(firm_returns.return_year == return_year) & (firm_returns.return_quarter == return_quarter)][['permco','date','ret']]
    rows = []
    for model_name, columns in predictor_sets.items():
        scaler = StandardScaler().fit(panel.loc[~panel.is_test, columns])
        X_all = scaler.transform(panel[columns])
        for month, monthly in outcome.groupby('date'):
            joined = panel[['permco','is_test']].merge(monthly, on='permco', how='inner')
            positions = panel.reset_index().set_index('permco').loc[joined.permco, 'index'].to_numpy()
            train = ~joined.is_test.to_numpy(); test = joined.is_test.to_numpy()
            if train.sum() < 100 or test.sum() < 25: continue
            coefficient, alpha = ridge_fit_cv(X_all[positions][train], joined.ret.to_numpy()[train], FIRM_SPLIT_SEED)
            prediction = np.column_stack([np.ones(test.sum()), X_all[positions][test]]) @ coefficient
            actual = joined.ret.to_numpy()[test]; denominator = np.var(actual)
            rows.append({'embedding_year':year,'embedding_quarter':quarter,'return_year':return_year,'return_quarter':return_quarter,'month':month,'model':model_name,'predictors':len(columns),'training_firms':int(train.sum()),'test_firms':int(test.sum()),'selected_alpha':alpha,'monthly_variance_r2':float(1-np.var(actual-prediction)/denominator) if denominator>0 else np.nan})
    result = pd.DataFrame(rows)
    save_parquet(result, checkpoint); quarter_results.append(result)
    print(label, '->', target_label, f'{time.perf_counter()-started:.1f}s')
    del panel, outcome, result; gc.collect()
return_comovement_results = pd.concat(quarter_results, ignore_index=True)
save_parquet(return_comovement_results, RC_ROOT / 'monthly_results.parquet')
return_comovement_summary = return_comovement_results.groupby('model', as_index=False).agg(return_comovement_r2=('monthly_variance_r2','mean'),monthly_r2_std=('monthly_variance_r2','std'),months=('monthly_variance_r2','count'),mean_selected_alpha=('selected_alpha','mean')).sort_values('return_comovement_r2', ascending=False)
save_parquet(return_comovement_summary, RC_ROOT / 'summary.parquet')
display(return_comovement_summary)

## 8. Visualize and audit the return-comovement results

In [ ]:
import matplotlib.pyplot as plt
plot = return_comovement_summary.sort_values('return_comovement_r2')
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(plot.model, 100 * plot.return_comovement_r2, color='#3b6ea8')
ax.axvline(0, color='black', linewidth=0.8); ax.set_xlabel('Mean monthly out-of-sample variance R² (%)')
ax.set_title('Return-comovement benchmark, 2005Q2–2024Q4')
fig.tight_layout(); fig.savefig(RC_ROOT / 'return_comovement_summary.png', dpi=180, bbox_inches='tight'); plt.show()
coverage = return_comovement_results.groupby('model').agg(months=('month','nunique'),quarters=('return_quarter','size'),minimum_test_firms=('test_firms','min'),nonfinite_r2=('monthly_variance_r2',lambda x:int((~np.isfinite(x)).sum()))).reset_index()
display(coverage)
assert coverage.months.min() == firm_returns[(firm_returns.date >= '2005-04-01')].date.nunique()
assert coverage.nonfinite_r2.eq(0).all()

## 9. Managed-portfolio benchmark: paper-style single-quarter design

Like the paper, the baseline managed-portfolio benchmark uses one cross-section. We use 2024Q4, split investors 90%/10%, fit models using training investors only, and mask the second-largest position of every test investor. Existing full-sample embeddings are not used for scoring.

In [ ]:
MP_YEAR, MP_QUARTER = 2024, 4
MP_LABEL = qlabel(MP_YEAR, MP_QUARTER)
MP_DIMENSIONS = [4, 10]
MP_CONTEXT_RADIUS = 10
MP_CONFIG = {'epochs':8,'batch_size':1024,'learning_rate':0.01,'validation_fraction':0.10,'seed':INVESTOR_SPLIT_SEED + 6000}
matrix_root = FULL_ROOT / 'matrices' / MP_LABEL
binary_mp = sparse.load_npz(matrix_root / 'binary.npz').tocsr().astype('float64')
log_mp = sparse.load_npz(matrix_root / 'log_dollar_holdings_positive.npz').tocsr().astype('float64')
ranks_mp = sparse.load_npz(FULL_ROOT / 'additional_recommender_systems' / 'matrices' / MP_LABEL / 'ranks.npz').tocsr().astype('float64')
investors_mp = pd.read_parquet(matrix_root / 'investor_mapping.parquet').sort_values('investor_index').reset_index(drop=True)
firms_mp = pd.read_parquet(matrix_root / 'firm_mapping.parquet').sort_values('firm_index').reset_index(drop=True)
rng = np.random.default_rng(INVESTOR_SPLIT_SEED)
test_count = int(np.ceil(0.10 * len(investors_mp))); order = rng.permutation(len(investors_mp))
test_indices = np.sort(order[:test_count]); train_indices = np.sort(order[test_count:])
def ordered_positions(row_index):
    start, end = log_mp.indptr[row_index], log_mp.indptr[row_index+1]
    columns = log_mp.indices[start:end]; values = log_mp.data[start:end]
    return columns[np.lexsort((columns, -values))]
test_sequences = {int(i): ordered_positions(int(i)) for i in test_indices}
eligible_test = np.array([i for i in test_indices if len(test_sequences[int(i)]) >= 3], dtype='int64')
split_audit = pd.DataFrame({'metric':['investors','training_investors','test_investors','eligible_test_investors','firms','positions'],'value':[len(investors_mp),len(train_indices),len(test_indices),len(eligible_test),len(firms_mp),binary_mp.nnz]})
save_parquet(split_audit, MP_ROOT / 'split_audit.parquet'); display(split_audit)

## 10. Training-only recommender systems and masked-position scores

RS-Binary, RS-Ranks, RS-L-0, and RS-L-Min are re-estimated by PCA on training investors. For the log-holdings models, investor and firm fixed effects are also estimated without test-investor information. Test-investor representations use only unmasked positions.

In [ ]:
from sklearn.decomposition import PCA
from scipy.special import logsumexp

def training_log_inputs():
    train = log_mp[train_indices].tocsr(); coo = train.tocoo()
    counts_i = np.bincount(coo.row, minlength=train.shape[0]); sums_i = np.bincount(coo.row, weights=coo.data, minlength=train.shape[0])
    investor_fe = sums_i / counts_i; demeaned = coo.data - investor_fe[coo.row]
    counts_a = np.bincount(coo.col, minlength=train.shape[1]); sums_a = np.bincount(coo.col, weights=demeaned, minlength=train.shape[1])
    firm_fe = np.divide(sums_a, counts_a, out=np.zeros_like(sums_a), where=counts_a>0)
    residual = demeaned - firm_fe[coo.col]
    l0 = sparse.csr_matrix((residual,(coo.row,coo.col)),shape=train.shape).toarray().astype('float32')
    minimum = np.full(train.shape[0], np.inf); np.minimum.at(minimum, coo.row, residual)
    lmin = np.repeat(minimum[:,None], train.shape[1], axis=1).astype('float32'); lmin[coo.row,coo.col] = residual.astype('float32')
    return l0, lmin, firm_fe

l0_train, lmin_train, mp_firm_fe = training_log_inputs()
training_inputs = {'rs_binary':binary_mp[train_indices].toarray().astype('float32'),'rs_ranks':ranks_mp[train_indices].toarray().astype('float32'),'rs_l_0':l0_train,'rs_l_min':lmin_train}
def masked_test_row(model, investor_index, masked):
    if model == 'rs_binary':
        row = binary_mp[investor_index].toarray().ravel(); row[masked] = 0; return row
    if model == 'rs_ranks':
        row = ranks_mp[investor_index].toarray().ravel(); row[masked] = 0; return row
    start,end = log_mp.indptr[investor_index],log_mp.indptr[investor_index+1]
    cols=log_mp.indices[start:end]; vals=log_mp.data[start:end]; keep=cols!=masked; cols=cols[keep]; vals=vals[keep]
    residual=vals-vals.mean()-mp_firm_fe[cols]
    if model == 'rs_l_0': row=np.zeros(log_mp.shape[1],dtype='float64')
    else: row=np.full(log_mp.shape[1],residual.min(),dtype='float64')
    row[cols]=residual; return row

def evaluate_logits(model, dimension, score_function):
    records=[]
    for investor_index in eligible_test:
        sequence=test_sequences[int(investor_index)]; target=int(sequence[1]); held=set(map(int,np.delete(sequence,1)))
        scores=np.asarray(score_function(int(investor_index),target,sequence),dtype='float64')
        candidate=np.ones(len(firms_mp),dtype=bool); candidate[list(held)]=False
        candidate_scores=scores[candidate]; target_score=scores[target]
        log_probability=float(target_score-logsumexp(candidate_scores)); candidate_count=int(candidate.sum())
        rank=int(1+np.sum(candidate_scores>target_score))
        records.append({'model':model,'dimension':dimension,'investor_index':int(investor_index),'target_firm_index':target,'candidate_count':candidate_count,'log_probability':log_probability,'random_log_probability':-np.log(candidate_count),'rank':rank})
    return pd.DataFrame(records)

mp_predictions=[]; mp_fit_rows=[]
for model,dense in training_inputs.items():
    for dimension in MP_DIMENSIONS:
        started=time.perf_counter(); pca=PCA(n_components=dimension,svd_solver='randomized',random_state=INVESTOR_SPLIT_SEED).fit(dense)
        def pca_scores(i,target,sequence,pca=pca,model=model):
            row=masked_test_row(model,i,target); return pca.inverse_transform(pca.transform(row.reshape(1,-1)))[0]
        prediction=evaluate_logits(model,dimension,pca_scores); mp_predictions.append(prediction)
        mp_fit_rows.append({'model':model,'dimension':dimension,'fit_seconds':time.perf_counter()-started,'training_investors':len(train_indices),'test_investors':len(prediction)})
        print(model,dimension,'complete')
        del pca; gc.collect()
del l0_train,lmin_train,training_inputs; gc.collect()
save_parquet(pd.DataFrame(mp_fit_rows),MP_ROOT/'pca_fit_audit.parquet')

## 11. Training-only Word2Vec and final managed-portfolio measure

Word2Vec is retrained using only training-investor sequences. For each masked second-largest position, its probability is computed from the surrounding portfolio-weight ranks, excluding already-held firms from the candidate set.

In [ ]:
import sys
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0,str(PROJECT_ROOT))
from src.paper_word2vec import fit_cbow, choose_device
existing_prediction_path=MP_ROOT/'masked_position_predictions.parquet'
if existing_prediction_path.exists():
    mp_predictions=[pd.read_parquet(existing_prediction_path).query("model != 'word2vec'").copy()]
all_sequences=[ordered_positions(i) for i in range(len(investors_mp))]
training_sequences=[all_sequences[i] for i in train_indices]
for dimension in MP_DIMENSIONS:
    directory=MP_ROOT/'word2vec_training_only_raw'/f'dimension={dimension}'; directory.mkdir(parents=True,exist_ok=True)
    raw_path=directory/'raw_cbow_weights.npy'
    if raw_path.exists() and not FORCE_REBUILD:
        values=np.load(raw_path).astype('float64')
    else:
        values,history,device=fit_cbow(training_sequences,len(firms_mp),dimension,MP_CONTEXT_RADIUS,MP_CONFIG['epochs'],MP_CONFIG['batch_size'],MP_CONFIG['learning_rate'],MP_CONFIG['validation_fraction'],MP_CONFIG['seed']+dimension,device=choose_device())
        np.save(raw_path,values); save_parquet(history,directory/'training_history.parquet')
        save_parquet(pd.DataFrame({'dimension':[dimension],'device':[device],'epochs_completed':[len(history)],'best_validation_loss':[history.validation_loss.min()]}),directory/'fit_summary.parquet')
    values=np.asarray(values,dtype='float64')
    def w2v_scores(i,target,sequence,values=values):
        position=1; left=max(0,position-MP_CONTEXT_RADIUS); right=min(len(sequence),position+MP_CONTEXT_RADIUS+1)
        context=np.concatenate([sequence[left:position],sequence[position+1:right]])
        return values @ values[context].mean(axis=0)
    prediction=evaluate_logits('word2vec',dimension,w2v_scores); mp_predictions.append(prediction); print('word2vec',dimension,'complete')
managed_predictions=pd.concat(mp_predictions,ignore_index=True)
save_parquet(managed_predictions,MP_ROOT/'masked_position_predictions.parquet')
managed_summary=managed_predictions.groupby(['model','dimension'],as_index=False).agg(test_investors=('investor_index','size'),mean_log_probability=('log_probability','mean'),mean_random_log_probability=('random_log_probability','mean'),top_1_accuracy=('rank',lambda x:np.mean(x<=1)),top_10_accuracy=('rank',lambda x:np.mean(x<=10)),median_rank=('rank','median'))
managed_summary['managed_portfolio_score']=1-managed_summary.mean_log_probability/managed_summary.mean_random_log_probability
managed_summary=managed_summary.sort_values('managed_portfolio_score',ascending=False)
save_parquet(managed_summary,MP_ROOT/'managed_portfolio_summary.parquet'); display(managed_summary)

## 12. Managed-portfolio visualization and integrity checks

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,5)); ordered=managed_summary.sort_values('managed_portfolio_score')
labels=ordered.model+' '+ordered.dimension.astype(str)+'D'
axes[0].barh(labels,100*ordered.managed_portfolio_score,color='#3b6ea8'); axes[0].axvline(0,color='black',linewidth=.8); axes[0].set_xlabel('Managed-portfolio score (%)')
axes[1].barh(labels,100*ordered.top_10_accuracy,color='#6a9f58'); axes[1].set_xlabel('Top-10 masked-position accuracy (%)')
fig.suptitle('Training-only managed-portfolio benchmark, 2024Q4'); fig.tight_layout(); fig.savefig(MP_ROOT/'managed_portfolio_summary.png',dpi=180,bbox_inches='tight'); plt.show()
assert managed_predictions.groupby(['model','dimension']).investor_index.nunique().eq(len(eligible_test)).all()
assert np.isfinite(managed_predictions.log_probability).all()
assert managed_predictions.candidate_count.min()>0
print('Managed-portfolio integrity checks passed.')

## 13. Add training-only RS-L by alternating least squares

RS-L is fitted only to positive log-holding residuals of training investors. The regularization is fixed at 10, the value selected in every quarter and dimension by the completed 80-quarter cross-validation audit. For each test investor, its factor exposure is inferred from unmasked positive positions while the asset factors and firm fixed effects remain fixed.

In [ ]:
RS_L_REGULARIZATION=10.0
RS_L_MAX_ITERATIONS=50
RS_L_TOLERANCE=1e-5

def training_residual_matrix():
    train=log_mp[train_indices].tocsr(); coo=train.tocoo()
    counts_i=np.bincount(coo.row,minlength=train.shape[0]); sums_i=np.bincount(coo.row,weights=coo.data,minlength=train.shape[0])
    investor_fe=sums_i/counts_i; demeaned=coo.data-investor_fe[coo.row]
    counts_a=np.bincount(coo.col,minlength=train.shape[1]); sums_a=np.bincount(coo.col,weights=demeaned,minlength=train.shape[1])
    firm_fe=np.divide(sums_a,counts_a,out=np.zeros_like(sums_a),where=counts_a>0)
    residual=demeaned-firm_fe[coo.col]
    return sparse.csr_matrix((residual,(coo.row,coo.col)),shape=train.shape),firm_fe

def als_update(source,fixed,regularization):
    source=source.tocsr(); dimension=fixed.shape[1]; identity=np.eye(dimension); updated=np.empty((source.shape[0],dimension))
    for row in range(source.shape[0]):
        start,end=source.indptr[row],source.indptr[row+1]; indices=source.indices[start:end]; values=source.data[start:end]; design=fixed[indices]
        updated[row]=np.linalg.solve(design.T@design+regularization*identity,design.T@values)
    return updated

def observed_rmse_fast(matrix,investors,assets):
    coo=matrix.tocoo(); prediction=np.sum(investors[coo.row]*assets[coo.col],axis=1)
    return float(np.sqrt(np.mean((coo.data-prediction)**2)))

def fit_training_rs_l(matrix,dimension,directory):
    asset_path=directory/'asset_factors.npy'; history_path=directory/'training_history.parquet'
    if asset_path.exists() and history_path.exists() and not FORCE_REBUILD:
        return np.load(asset_path),pd.read_parquet(history_path)
    rng=np.random.default_rng(INVESTOR_SPLIT_SEED+dimension); assets=rng.normal(scale=.05,size=(matrix.shape[1],dimension)); investors=rng.normal(scale=.05,size=(matrix.shape[0],dimension)); history=[]; previous=None
    for iteration in range(1,RS_L_MAX_ITERATIONS+1):
        investors=als_update(matrix,assets,RS_L_REGULARIZATION)
        assets=als_update(matrix.T.tocsr(),investors,RS_L_REGULARIZATION)
        rmse=observed_rmse_fast(matrix,investors,assets); history.append({'iteration':iteration,'observed_rmse':rmse})
        print(f'RS-L {dimension}D iteration {iteration}: RMSE {rmse:.6f}')
        if previous is not None and abs(previous-rmse)<=RS_L_TOLERANCE*max(abs(previous),1.0): break
        previous=rmse
    directory.mkdir(parents=True,exist_ok=True); np.save(asset_path,assets); save_parquet(pd.DataFrame(history),history_path)
    return assets,pd.DataFrame(history)

rs_l_matrix,rs_l_firm_fe=training_residual_matrix()
existing=pd.read_parquet(MP_ROOT/'masked_position_predictions.parquet').query("model != 'rs_l'").copy()
prediction_parts=[existing]; rs_l_fit_rows=[]
for dimension in MP_DIMENSIONS:
    started=time.perf_counter(); directory=MP_ROOT/'rs_l_training_only'/f'dimension={dimension}'
    assets,history=fit_training_rs_l(rs_l_matrix,dimension,directory)
    identity=np.eye(dimension)
    def rs_l_scores(i,target,sequence,assets=assets):
        start,end=log_mp.indptr[i],log_mp.indptr[i+1]; cols=log_mp.indices[start:end]; vals=log_mp.data[start:end]; keep=cols!=target; cols=cols[keep]; vals=vals[keep]
        residual=vals-vals.mean()-rs_l_firm_fe[cols]; design=assets[cols]
        investor=np.linalg.solve(design.T@design+RS_L_REGULARIZATION*identity,design.T@residual)
        return assets@investor+rs_l_firm_fe
    prediction=evaluate_logits('rs_l',dimension,rs_l_scores); prediction_parts.append(prediction)
    rs_l_fit_rows.append({'model':'rs_l','dimension':dimension,'regularization':RS_L_REGULARIZATION,'iterations':len(history),'observed_rmse':float(history.iloc[-1].observed_rmse),'fit_seconds':time.perf_counter()-started,'training_investors':len(train_indices),'test_investors':len(prediction)})
managed_predictions=pd.concat(prediction_parts,ignore_index=True); save_parquet(managed_predictions,MP_ROOT/'masked_position_predictions.parquet')
save_parquet(pd.DataFrame(rs_l_fit_rows),MP_ROOT/'rs_l_fit_audit.parquet')
managed_summary=managed_predictions.groupby(['model','dimension'],as_index=False).agg(test_investors=('investor_index','size'),mean_log_probability=('log_probability','mean'),mean_random_log_probability=('random_log_probability','mean'),top_1_accuracy=('rank',lambda x:np.mean(x<=1)),top_10_accuracy=('rank',lambda x:np.mean(x<=10)),median_rank=('rank','median'))
managed_summary['managed_portfolio_score']=1-managed_summary.mean_log_probability/managed_summary.mean_random_log_probability
managed_summary=managed_summary.sort_values('managed_portfolio_score',ascending=False); save_parquet(managed_summary,MP_ROOT/'managed_portfolio_summary.parquet'); display(managed_summary)
del rs_l_matrix; gc.collect()

## 14. Final managed-portfolio comparison and completion audit

In [ ]:
ordered=managed_summary.sort_values('managed_portfolio_score'); labels=ordered.model+' '+ordered.dimension.astype(str)+'D'
fig,axes=plt.subplots(1,2,figsize=(13,6)); axes[0].barh(labels,100*ordered.managed_portfolio_score,color='#3b6ea8'); axes[0].axvline(0,color='black',linewidth=.8); axes[0].set_xlabel('Managed-portfolio score (%)')
axes[1].barh(labels,100*ordered.top_10_accuracy,color='#6a9f58'); axes[1].set_xlabel('Top-10 accuracy (%)'); fig.suptitle('Complete managed-portfolio benchmark, 2024Q4'); fig.tight_layout(); fig.savefig(MP_ROOT/'managed_portfolio_complete.png',dpi=180,bbox_inches='tight'); plt.show()
expected={(model,dimension) for model in ['rs_binary','rs_ranks','rs_l_0','rs_l_min','rs_l','word2vec'] for dimension in MP_DIMENSIONS}
observed=set(map(tuple,managed_summary[['model','dimension']].to_numpy()))
assert observed==expected,f'Missing specifications: {expected-observed}'
assert managed_predictions.groupby(['model','dimension']).investor_index.nunique().eq(len(eligible_test)).all()
assert np.isfinite(managed_predictions.log_probability).all()
completion=pd.DataFrame({'benchmark':['relative_valuation','return_comovement','managed_portfolio'],'status':['complete','complete','complete']})
save_parquet(completion,BENCH_ROOT/'replication_benchmark_completion.parquet'); display(completion)